# 04 · BANKSY-aware integration

Uses the full 1500-feature BANKSY objects (`_banksy_full.h5ad`) from notebook 01.
Unlike notebook 02 (which uses scVI on 500 genes), this notebook integrates cells
using both their own expression AND their spatial neighbourhood signal.

**Why this is different from notebook 02:**
- scVI operates on 500 raw count genes only
- This pipeline operates on 1500 BANKSY features (500 genes + 1000 neighbourhood features)
- Cells are separated not just by their own expression but by what surrounds them spatially
- Harmony is used instead of scVI because BANKSY features are continuous (not counts)

```
BANKSY_DIR/<section>/<section>_banksy_full.h5ad
        ↓
Concatenate all sections (1500 features)
        ↓
Regress out total_counts
        ↓
PCA on 1500 features
        ↓
Harmony batch correction (batch = section)
        ↓
neighbors → UMAP → Leiden
        ↓
SAVE_PATH
```

## 1 · Imports

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scanpy.external as sce
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")
sc.logging.print_header()

## 2 · Paths & parameters
**Edit this cell.**

In [ ]:
BANKSY_DIR = "/path/to/banksy/output/"
SAVE_PATH  = "/path/to/output/banksy_aware_integrated.h5ad"

PARAMS = dict(
    seed               = 1234,
    batch_key          = "section",

    # ── PCA ─────────────────────────────────────────────────────────────────
    n_pcs              = 50,    # more PCs than usual — 1500 features have more variance

    # ── Harmony ─────────────────────────────────────────────────────────────
    max_iter_harmony   = 50,

    # ── Clustering ──────────────────────────────────────────────────────────
    n_neighbors        = 30,
    leiden_resolutions = [3.5],
    umap_min_dist      = 0.001,
    n_plot_cells       = 100000,

    # ── Section exclusion ───────────────────────────────────────────────────
    min_cells_section  = 100,
)

## 3 · Helper functions

### 3.1 · Environment

In [ ]:
def setup_environment(seed):
    np.random.seed(seed); random.seed(seed)
    sc.set_figure_params(facecolor="white", figsize=(8, 8))
    sc.settings.verbosity = 1
    sns.set_style("white")

### 3.2 · Load & concatenate BANKSY full objects

In [ ]:
def load_and_concatenate(banksy_dir):
    """
    Scan banksy_dir for *_banksy_full.h5ad files (1500 features per section).
    Concatenate into one AnnData with:
      obs['section']  = section name
      obs_names       = <section>_<barcode>  (unique across all sections)
    Sets adata.X = transformed layer (log-normalized — correct input for PCA).
    """
    paths = []
    for section in sorted(os.listdir(banksy_dir)):
        sec_dir = os.path.join(banksy_dir, section)
        if not os.path.isdir(sec_dir):
            continue
        for f in os.listdir(sec_dir):
            if f.endswith("_banksy_full.h5ad"):
                paths.append((section, os.path.join(sec_dir, f)))

    if not paths:
        raise FileNotFoundError(f"No *_banksy_full.h5ad files found in {banksy_dir}")

    print(f"  Found {len(paths)} sections")

    adata_list = []
    for section, path in paths:
        ad = sc.read_h5ad(path)
        ad.obs_names  = [f"{section}_{bc}" for bc in ad.obs_names]
        ad.obs["section"] = section

        # Use transformed layer (log-normalized) — NOT raw counts
        # BANKSY neighbourhood features are continuous, not counts
        if "transformed" in ad.layers:
            ad.X = ad.layers["transformed"].copy()
        if not sp.isspmatrix_csr(ad.X):
            ad.X = sp.csr_matrix(ad.X)

        adata_list.append(ad)
        print(f"    {section:<40} {ad.n_obs:,} cells x {ad.shape[1]:,} features")

    adata = sc.concat(adata_list, join="inner", merge="same")
    print(f"\n  Concatenated : {adata.shape[0]:,} cells x {adata.shape[1]:,} features")
    print(f"  Sections     : {adata.obs['section'].nunique()}")
    print(f"  obs_names unique: {adata.obs_names.is_unique}")
    return adata

### 3.3 · Exclude sparse sections

In [ ]:
def exclude_sparse_sections(adata):
    """Remove sections with fewer than min_cells_section cells."""
    min_cells = PARAMS["min_cells_section"]
    counts    = adata.obs["section"].value_counts()
    exclude   = counts[counts < min_cells].index.tolist()

    if exclude:
        print(f"  Excluding {len(exclude)} section(s) with < {min_cells} cells:")
        for s in exclude:
            print(f"    {s} — {counts[s]} cells")
        adata = adata[~adata.obs["section"].isin(exclude)].copy()
    else:
        print(f"  All sections have >= {min_cells} cells — none excluded")

    print(f"  After exclusion: {adata.n_obs:,} cells | "
          f"{adata.obs['section'].nunique()} sections")
    return adata

### 3.4 · QC

In [ ]:
def run_qc(adata):
    """
    Compute QC metrics and plot distributions.
    For BANKSY full objects, n_genes_by_counts reflects the 500 original genes
    (BANKSY neighbourhood features are not counts and won't affect this).
    """
    # QC on original genes only (first 500 columns)
    sc.pp.calculate_qc_metrics(
        adata, percent_top=None, log1p=False, inplace=True
    )
    print(f"  Cells: {adata.n_obs:,}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(adata.obs["n_genes_by_counts"], bins=100)
    axes[0].set_xlabel("Genes per cell")
    axes[0].set_title("Gene count distribution")

    axes[1].hist(adata.obs["total_counts"], bins=100)
    axes[1].set_xlabel("Total counts per cell")
    axes[1].set_title("Total count distribution")

    plt.suptitle("QC metrics")
    plt.tight_layout()
    plt.show()
    return adata

### 3.5 · Regress out total_counts

In [ ]:
def regress_total_counts(adata):
    """
    Regress total_counts out of all 1500 BANKSY features.
    This removes library size bias before PCA.
    """
    print(f"  Regressing total_counts out of {adata.shape[1]:,} features...")
    X            = adata.X.toarray() if sp.issparse(adata.X) else adata.X.copy()
    total_counts = adata.obs["total_counts"].values.reshape(-1, 1)

    X_regressed = np.zeros_like(X)
    for dim in range(X.shape[1]):
        reg = LinearRegression().fit(total_counts, X[:, dim])
        X_regressed[:, dim] = X[:, dim] - reg.predict(total_counts)

    adata.X = sp.csr_matrix(X_regressed)
    adata.layers["banksy_regressed"] = adata.X.copy()
    print("  Done")
    return adata

### 3.6 · PCA + Harmony

In [ ]:
def run_pca_harmony(adata):
    """
    Run PCA on the 1500 BANKSY features then Harmony for batch correction.
    Adds:
      obsm['X_pca']
      obsm['X_pca_harmony']
    """
    print(f"  Running PCA (n_comps={PARAMS['n_pcs']})...")
    sc.tl.pca(adata, n_comps=PARAMS["n_pcs"], svd_solver="arpack")

    print("  Running Harmony batch correction...")
    sce.pp.harmony_integrate(
        adata,
        key              = PARAMS["batch_key"],
        basis            = "X_pca",
        max_iter_harmony = PARAMS["max_iter_harmony"],
    )
    print(f"  Done — harmony embedding: {adata.obsm['X_pca_harmony'].shape}")

### 3.7 · Cluster & UMAP

In [ ]:
def cluster_and_embed(adata):
    """
    Build neighbourhood graph on X_pca_harmony, compute UMAP and Leiden clusters.
    Adds:
      obsm['X_umap_banksy']
      obs['leiden_<resolution>'] for each resolution
    """
    print("  Building neighbourhood graph on X_pca_harmony...")
    sc.pp.neighbors(adata, use_rep="X_pca_harmony",
                    n_neighbors=PARAMS["n_neighbors"])

    print("  Computing UMAP...")
    sc.tl.umap(adata, min_dist=PARAMS["umap_min_dist"])
    adata.obsm["X_umap_banksy"] = adata.obsm["X_umap"].copy()

    leiden_cols = []
    for r in PARAMS["leiden_resolutions"]:
        col = f"leiden_{r:g}"
        sc.tl.leiden(
            adata,
            resolution   = float(r),
            key_added    = col,
            flavor       = "igraph",
            n_iterations = 2,
        )
        n_cl = adata.obs[col].nunique()
        print(f"  resolution={r} → {n_cl} clusters")
        leiden_cols.append(col)

    # Plot on subsample
    n_plot     = PARAMS["n_plot_cells"]
    idx        = np.random.choice(adata.n_obs, min(n_plot, adata.n_obs), replace=False)
    adata_plot = adata[idx].copy()
    print(f"  Plotting UMAP on {len(idx):,} cells...")
    sc.pl.umap(adata_plot, color=leiden_cols + ["section", "total_counts"],
               ncols=3)

### 3.8 · Save

In [ ]:
def save_object(adata, save_path):
    """Save the BANKSY-aware integrated object."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Set X = transformed for downstream use
    if "transformed" in adata.layers:
        adata.X = adata.layers["transformed"].copy()
        if sp.issparse(adata.X):
            adata.X = adata.X.tocsr()

    adata.write(save_path)
    print(f"  [✓] Saved → {save_path}")
    print(f"      {adata.shape[0]:,} cells x {adata.shape[1]:,} features")
    print(f"      layers : {list(adata.layers.keys())}")
    print(f"      obsm   : {list(adata.obsm.keys())}")
    print(f"      obs    : {list(adata.obs.columns)}")

## 4 · Run

In [ ]:
setup_environment(PARAMS["seed"])

print("[1/6] Loading & concatenating BANKSY full objects...")
adata = load_and_concatenate(BANKSY_DIR)

In [ ]:
print("\n[2/6] Excluding sparse sections...")
adata = exclude_sparse_sections(adata)

In [ ]:
print("\n[3/6] QC...")
adata = run_qc(adata)

In [ ]:
print("\n[4/6] Regressing out total_counts...")
adata = regress_total_counts(adata)

In [ ]:
print("\n[5/6] PCA + Harmony...")
run_pca_harmony(adata)

In [ ]:
print("\n[6/6] Clustering & UMAP...")
cluster_and_embed(adata)

print("\nSaving...")
save_object(adata, SAVE_PATH)
print("Done!")

### Add new resolutions

In [ ]:
adata = sc.read_h5ad("/path/to/your/file.h5ad")

In [ ]:
# Add new resolutions without recomputing neighbors or UMAP
new_resolutions = [0.5]

leiden_cols = []
for r in new_resolutions:
    col = f"leiden_{r:g}"
    sc.tl.leiden(
        adata,
        resolution   = float(r),
        key_added    = col,
        flavor       = "igraph",
        n_iterations = 2,
    )
    print(f"  resolution={r} → {adata.obs[col].nunique()} clusters")
    leiden_cols.append(col)

# Plot on subsample
n_plot     = PARAMS["n_plot_cells"]
idx        = np.random.choice(adata.n_obs, min(n_plot, adata.n_obs), replace=False)
adata_plot = adata[idx].copy()
adata_plot.obsm["X_umap"] = adata_plot.obsm["X_umap_banksy"].copy()
sc.pl.umap(adata_plot, color=leiden_cols, ncols=3)

In [ ]:
SAVE_DIR = "/path/to/your/figures/directory/"

import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42   # embeds fonts as text, not outlines
mpl.rcParams["ps.fonttype"]  = 42

color_map = {
    '0' : "#B39DFA",  # neurons
    '1' : "#3ECD3E",  # astrocytes
    '2' : "#3ECD3E",  # astrocytes
    '3' : "#808080",  # Vascular/perivascular
    '4' : "#0C0CFF",  # oligo
    '5' : "#FBBF24",  # Immune cells
    '6' : "#2EC6FE",  # OPC
    '7' : "#FF0606",  # Perivascular fibro
    '8' : "#FF0606",  # Fibro
    '9' : "#3ECD3E",  # astrocytes
    '10': "#FF0606",  # Fibro
    '11': "#808080" # Endo
}

sc.pl.umap(adata, color='leiden_0.5', ncols=3, show=False)
plt.savefig(
    os.path.join(SAVE_DIR, "umap_leiden.pdf"),
    bbox_inches="tight", dpi=300, format="pdf"
)
plt.show()

In [ ]:
if "leiden_0.5_colors" in adata.uns:
    del adata.uns["leiden_0.5_colors"]

sc.pl.umap(adata, color="leiden_0.5", show=False)
plt.savefig(
    os.path.join(SAVE_DIR, "umap_leiden.pdf"),
    bbox_inches="tight", dpi=300, format="pdf"
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

markers = ['Hexb', 'Itgam', "Ptprc", "Aif1", "Lgals3", "Csf1r", "Col1a2", "Postn", "Col3a1", "Vwf", "Cdh5", "Cldn5", "Gabarapl1", "Rbfox3", "Thy1", "Aldh1a1", "Gja1", "Sox9", "Foxj1",
          "Sox10", "Olig2", "Pdgfra", "Pdgfrb", "Rgs5"]


import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

dp = sc.pl.dotplot(
    adata,
    markers,
    groupby="leiden_0.5",
    standard_scale="var",
    dot_min=0.05,
    figsize=(10, 7),
    return_fig=True,
    show=False,
)

dp.make_figure()  # force rendering
fig = plt.gcf()   # grab the real current figure
plt.savefig(
    os.path.join(SAVE_DIR, "dotplot_banksy.pdf"),
    bbox_inches="tight", dpi=300, format="pdf"
)
plt.show()

In [ ]:
genes_of_interest = ['Vwf']

sc.pl.umap(adata, color=genes_of_interest,
           frameon=False, legend_loc='right margin',
           ncols=4, cmap='Reds', size = 20)

In [ ]:
# Use full 1500-feature object directly
adata.X = adata.X.copy()  # already log-normalized

sc.tl.rank_genes_groups(
    adata,
    groupby   = "leiden_0.5",    # change to your resolution
    method    = "wilcoxon",
    use_raw   = False,
    key_added = "rank_features_1500",
)

# Results table
deg_df = sc.get.rank_genes_groups_df(
    adata,
    group        = None,
    key          = "rank_features_1500",
    pval_cutoff  = 0.05,
)
print(f"Significant features: {len(deg_df):,}")

# Split results into genes vs neighbourhood features
deg_df["is_nbr"] = adata.var.loc[deg_df["names"], "is_nbr"].values
deg_genes = deg_df[deg_df["is_nbr"] == False]
deg_nbr   = deg_df[deg_df["is_nbr"] == True]

print(f"\nDifferential genes              : {len(deg_genes):,}")
print(f"Differential neighbourhood features: {len(deg_nbr):,}")



In [ ]:
deg_df = sc.get.rank_genes_groups_df(
    adata,
    group        = None,
    key          = "rank_features_1500",
    pval_cutoff  = 0.05,
)


In [ ]:
import matplotlib as mpl
mpl.rcParams["pdf.fonttype"] = 42   # embeds fonts as text, not outlines
mpl.rcParams["ps.fonttype"]  = 42

SAVE_DIR = "/path/to/your/figures/directory/"

dp = sc.pl.rank_genes_groups_dotplot(
    adata,
    key            = "rank_features_1500",
    groupby        = "leiden_0.5",
    dendrogram     = False,
    standard_scale = "var",
    dot_min        = 0.1,
    var_names      = deg_df.groupby("group")
                           .head(3)["names"]
                           .unique()
                           .tolist(),
    return_fig     = True,
    show           = False,
)

dp.make_figure()
fig = plt.gcf()

# ── Save ──────────────────────────────────────────────────────────────────────
save_path = os.path.join(SAVE_DIR, "dotplot_immune_subset_banksy.pdf")
fig.savefig(
    save_path,
    bbox_inches = "tight",
    dpi         = 300,
    format      = "pdf",
    backend     = "pdf",
)
plt.show()
print(f"Saved → {save_path}")

In [ ]:
import matplotlib.pyplot as plt

markers = ['Hexb', 'Itgam', "Ptprc", "Aif1", "Lgals3", "Csf1r", "Col1a2", "Postn", "Col3a1", "Vwf", "Cdh5", "Cldn5", "Gabarapl1", "Rbfox3", "Thy1", "Aldh1a1", "Gja1", "Sox9", "Foxj1",
          "Sox10", "Olig2", "Pdgfra", "Cxcl14", "Zeb1"]


import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

dp = sc.pl.dotplot(
    adata,
    markers,
    groupby="meta_cluster",
    standard_scale="var",
    dot_min=0.05,
    figsize=(10, 7),
    return_fig=True,
    show=False,
)

dp.make_figure()  # force rendering
fig = plt.gcf()   # grab the real current figure


## Transfert cluster annotation to the banksy object

In [ ]:
non_banksy_adata = sc.read_h5ad("/path/to/your/annotated_cells.h5ad")

In [ ]:
# Initialize all cells as Unknown
adata.obs["meta_cluster"] = "Unknown"

# Find cells that exist in both objects
shared_cells = adata.obs_names.intersection(non_banksy_adata.obs_names)
print(f"Shared cells: {len(shared_cells):,} / {adata.n_obs:,}")

# Transfer meta_cluster labels for immune cells
adata.obs.loc[shared_cells, "meta_cluster"] = (
    non_banksy_adata.obs.loc[shared_cells, "meta_cluster"]
)

# Check result
print("\nMeta-cluster counts:")
print(adata.obs["meta_cluster"].value_counts().to_string())

# Plot
idx        = np.random.choice(adata.n_obs, min(100000, adata.n_obs), replace=False)
adata_plot = adata[idx].copy()
adata_plot.obsm["X_umap"] = adata_plot.obsm["X_umap_banksy"].copy()
sc.pl.umap(adata_plot, color="meta_cluster", legend_fontsize=8)

In [ ]:
#### import matplotlib.pyplot as plt
import numpy as np

cluster_col = "meta_cluster"

categories = adata.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 6                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = adata.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = adata.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = adata.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ── Subset annotated cells (non-Unknown) ─────────────────────────────────────
adata_annotated = adata[adata.obs["meta_cluster"] != "Unknown"].copy()
print(f"Annotated cells : {adata_annotated.n_obs:,}")
print(f"Sections        : {adata_annotated.obs['section'].nunique()}")
print(f"Meta-clusters   : {adata_annotated.obs['meta_cluster'].nunique()}")
print(adata_annotated.obs["meta_cluster"].value_counts().to_string())


## Reclustering with banksy features (With 12 clusters not 11)

In [ ]:
# ── Recluster on 1500 BANKSY features ────────────────────────────────────────
print("\nRunning PCA on 1500 features...")
sc.tl.pca(adata_annotated, n_comps=50, svd_solver="arpack")

print("Running Harmony batch correction...")
import scanpy.external as sce
sce.pp.harmony_integrate(
    adata_annotated,
    key              = "section",
    basis            = "X_pca",
    max_iter_harmony = 50,
)

print("Building neighbourhood graph...")
sc.pp.neighbors(adata_annotated, use_rep="X_pca_harmony", n_neighbors=20)

print("Computing UMAP...")
sc.tl.umap(adata_annotated, min_dist=0.1)
adata_annotated.obsm["X_umap_banksy"] = adata_annotated.obsm["X_umap"].copy()

print("Leiden clustering...")
for r in [0.8]:
    col = f"leiden_{r:g}"
    sc.tl.leiden(
        adata_annotated,
        resolution   = float(r),
        key_added    = col,
        flavor       = "igraph",
        n_iterations = 2,
    )
    print(f"  resolution={r} → {adata_annotated.obs[col].nunique()} clusters")

# ── Plot ──────────────────────────────────────────────────────────────────────
idx        = np.random.choice(adata_annotated.n_obs,
                              min(100000, adata_annotated.n_obs), replace=False)
adata_plot = adata_annotated[idx].copy()
sc.pl.umap(
    adata_plot,
    color  = ["meta_cluster", "leiden_0.8"],
    ncols  = 2,
    size   = 2,
)

In [ ]:
#### import matplotlib.pyplot as plt
import numpy as np

cluster_col = "meta_cluster"

categories = adata_annotated.obs[cluster_col].astype("category").cat.categories
n = len(categories)

# ---- Auto grid size ----
ncols = 5                              # change to control width
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

X = adata_annotated.obsm["X_umap"]

for i, cl in enumerate(categories):

    ax = axes[i]

    # Background cells
    bg = adata_annotated.obs[cluster_col] != cl
    ax.scatter(
        X[bg, 0],
        X[bg, 1],
        s=0.5,
        c="lightgray",
        alpha=0.3,
        linewidths=0
    )

    # Highlight cluster
    fg = adata_annotated.obs[cluster_col] == cl
    ax.scatter(
        X[fg, 0],
        X[fg, 1],
        s=1,              # make highlighted clearly larger
        c="red",
        alpha=0.9,
        linewidths=0
    )

    ax.set_title(f"Cluster {cl}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Hide empty panels
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

### Clustering only with expression and nbr_1 -> does not work

In [ ]:
# Keep only original genes (is_nbr=False) + nbr_1 features (k=1)
mask_features = (
    (adata_annotated.var["k"] == -1) |   # original genes
    (adata_annotated.var["k"] ==  1)     # nbr_1 features
)
adata_IC_nbr1 = adata_annotated[:, mask_features].copy()
print(f"Features kept: {adata_IC_nbr1.n_vars} "
      f"(genes: {(adata_IC_nbr1.var['is_nbr']==False).sum()}, "
      f"nbr_1: {(adata_IC_nbr1.var['k']==1).sum()})")

# ── PCA + Harmony + neighbors + UMAP + Leiden ────────────────────────────────
print("\nRunning PCA...")
sc.tl.pca(adata_IC_nbr1, n_comps=50, svd_solver="arpack")

print("Running Harmony...")
sce.pp.harmony_integrate(
    adata_IC_nbr1, key="section", basis="X_pca", max_iter_harmony=50
)

print("Building neighbours...")
sc.pp.neighbors(adata_IC_nbr1, use_rep="X_pca_harmony", n_neighbors=20)

print("Computing UMAP...")
sc.tl.umap(adata_IC_nbr1, min_dist=0.1)
adata_IC_nbr1.obsm["X_umap_genes_nbr1"] = adata_IC_nbr1.obsm["X_umap"].copy()

print("Leiden clustering...")
for r in [0.8]:
    col = f"leiden_{r:g}"
    sc.tl.leiden(
        adata_IC_nbr1, resolution=float(r), key_added=col,
        flavor="igraph", n_iterations=2,
    )
    print(f"  resolution={r} → {adata_IC_nbr1.obs[col].nunique()} clusters")

# ── Plot ──────────────────────────────────────────────────────────────────────
idx        = np.random.choice(adata_IC_nbr1.n_obs, min(100000, adata_IC_nbr1.n_obs), replace=False)
adata_IC_nbr1_plot = adata_IC_nbr1[idx].copy()
sc.pl.umap(adata_IC_nbr1_plot, color=["meta_cluster", "leiden_0.8"], ncols=3)

In [ ]:
sc.pl.umap(adata_IC_nbr1_plot, color=["meta_cluster", "leiden_0.8"], ncols=3)

### Clustering only with expression and nbr_0 -> same as with 1500 features

In [ ]:
mask_features = (
    (adata_annotated.var["k"] == -1) |   # original genes
    (adata_annotated.var["k"] ==  0)     # nbr_0 features
)
adata_IC_nbr0 = adata_annotated[:, mask_features].copy()
print(f"Features kept: {adata_IC_nbr0.n_vars} "
      f"(genes: {(adata_IC_nbr0.var['k'] == -1).sum()}, "
      f"nbr_0: {(adata_IC_nbr0.var['k'] ==  0).sum()})")

# ── PCA + Harmony + neighbors + UMAP + Leiden ────────────────────────────────
print("\nRunning PCA...")
sc.tl.pca(adata_IC_nbr0, n_comps=50, svd_solver="arpack")

print("Running Harmony...")
sce.pp.harmony_integrate(
    adata_IC_nbr0, key="section", basis="X_pca", max_iter_harmony=50
)

print("Building neighbours...")
sc.pp.neighbors(adata_IC_nbr0, use_rep="X_pca_harmony", n_neighbors=20)

print("Computing UMAP...")
sc.tl.umap(adata_IC_nbr0, min_dist=0.001)
adata_IC_nbr0.obsm["X_umap_genes_nbr0"] = adata_IC_nbr0.obsm["X_umap"].copy()

print("Leiden clustering...")
for r in [0.8]:
    col = f"leiden_{r:g}"
    sc.tl.leiden(
        adata_IC_nbr0, resolution=float(r), key_added=col,
        flavor="igraph", n_iterations=2,
    )
    print(f"  resolution={r} → {adata_IC_nbr0.obs[col].nunique()} clusters")

# ── Plot ──────────────────────────────────────────────────────────────────────
idx             = np.random.choice(adata_IC_nbr0.n_obs,
                                   min(100000, adata_IC_nbr0.n_obs), replace=False)
adata_IC_nbr0_plot = adata_IC_nbr0[idx].copy()
sc.pl.umap(adata_IC_nbr0_plot, color=["meta_cluster", "leiden_0.8"], ncols=2)

## Spatial map of meta_clusters

In [ ]:
# Sections to use: "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1", "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2", 
#                  "myelin_d1_1", "myelin_d3_4", "myelin_d4_3", "myelin_d5_1", "myelin_d7_11", "myelin_d9_9", "myelin_d14_11", 
#                  "liver_d1_2", "liver_d3_4", "liver_d4_1", "liver_d5_6", "liver_d7_14", "liver_d9_11", "liver_d14_11"


In [ ]:
adata

In [ ]:
sections_of_interest = [
    "uninjured_1", "crush_d1_3", "crush_d3_3", "crush_d4_1",
    "crush_d5_8", "crush_d7_1", "crush_d9_1", "crush_d14_2"
]

SAVE_DIR = "/path/to/your/figures/directory/"

meta_clusters = sorted(adata.obs["meta_cluster"][
    adata.obs["meta_cluster"] != "Unknown"
].unique())

for meta in meta_clusters:
    fig, axes = plt.subplots(
        1, len(sections_of_interest),
        figsize=(4 * len(sections_of_interest), 4)
    )
    fig.suptitle(f"Meta-cluster: {meta}", fontsize=14, y=1.02)

    for ax, section in zip(axes, sections_of_interest):
        # Subset to this section
        mask_sec = adata.obs["section"] == section
        adata_sec = adata[mask_sec]

        if adata_sec.n_obs == 0:
            ax.set_title(f"{section}\n(no cells)")
            ax.axis("off")
            continue

        # Split into target and other cells
        mask_meta  = adata_sec.obs["meta_cluster"] == meta
        mask_other = ~mask_meta

        # Plot all other cells in grey
        ax.scatter(
            adata_sec.obs.loc[mask_other, "x"],
            adata_sec.obs.loc[mask_other, "y"],
            s=0.5, c="lightgrey", alpha=0.3, rasterized=True
        )
        # Plot meta_cluster cells in red
        ax.scatter(
            adata_sec.obs.loc[mask_meta, "x"],
            adata_sec.obs.loc[mask_meta, "y"],
            s=1.0, c="crimson", alpha=0.8, rasterized=True
        )

        n_cells = mask_meta.sum()
        pct     = n_cells / adata_sec.n_obs * 100
        ax.set_title(f"{section}\n{n_cells:,} cells ({pct:.1f}%)", fontsize=8)
        ax.set_xlabel("x", fontsize=7)
        ax.set_ylabel("y", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.invert_yaxis()
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(
        os.path.join(SAVE_DIR, f"spatial_{meta.replace('/', '_').replace(' ', '_')}.pdf"),
        dpi=150, bbox_inches="tight"
    )
    plt.show()
    print(f"  Saved: {meta}")

In [ ]:
# ── Build composition table ───────────────────────────────────────────────────
adata_ann = adata[adata.obs["meta_cluster"] != "Unknown"].copy()

counts = (
    adata_ann.obs
    .groupby(["day", "meta_cluster"])
    .size()
    .unstack(fill_value=0)
)

comp_pct = counts.div(counts.sum(axis=1), axis=0) * 100

# ── Sort days and meta-clusters ───────────────────────────────────────────────
day_order  = ["uninjured", "d1", "d2", "d3", "d4", "d5", "d7", "d9", "d14"]
day_order  = [d for d in day_order if d in comp_pct.index]
comp_pct   = comp_pct.reindex(day_order)

meta_order = sorted(comp_pct.columns.tolist(),
                    key=lambda x: int(x.split("_")[-1]))

# Consistent colors across all plots
colors = dict(zip(meta_order, plt.cm.tab20.colors[:len(meta_order)]))

# ── One plot per timepoint ────────────────────────────────────────────────────
n_cols = 3
n_rows = int(np.ceil(len(day_order) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(5 * n_cols, 4 * n_rows),
                          sharey=True)
axes = axes.flatten()

for i, day in enumerate(day_order):
    ax    = axes[i]
    data  = comp_pct.loc[day].reindex(meta_order)
    total = int(counts.loc[day].sum())

    ax.bar(
        range(len(data)),
        data.values,
        color     = [colors[m] for m in data.index],
        edgecolor = "white",
        linewidth = 0.5,
    )
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=90, ha="right", fontsize=7)
    ax.set_title(f"{day}\n(n={total:,} immune cells)", fontsize=10)
    ax.set_ylabel("% of immune cells", fontsize=8)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v:.0f}%")
    )

# Turn off unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Immune cell composition per timepoint", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "immune_composition_per_timepoint.png"),
            dpi=180, bbox_inches="tight")
plt.show()